<a href="https://colab.research.google.com/github/Ashutosh-Mi/GenAI_ML/blob/main/TaskManager_withUserAuthentication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import hashlib
import os

USERS_FILE = "users.txt"


# --- Utility Functions ---

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()


def user_exists(username):
    if not os.path.exists(USERS_FILE):
        return False
    with open(USERS_FILE, "r") as f:
        for line in f:
            if line.split(",")[0] == username:
                return True
    return False


def get_tasks_file(username):
    return f"tasks_{username}.txt"


# --- Authentication ---

def register():
    username = input("Enter new username: ")
    if user_exists(username):
        print("Username already exists. Please try again.")
        return None
    password = input("Enter new password: ")
    hashed_pw = hash_password(password)

    with open(USERS_FILE, "a") as f:
        f.write(f"{username},{hashed_pw}\n")
    print("Registration successful.")
    return username


def login():
    username = input("Username: ")
    password = input("Password: ")
    hashed_pw = hash_password(password)

    if not os.path.exists(USERS_FILE):
        print("No users registered yet.")
        return None

    with open(USERS_FILE, "r") as f:
        for line in f:
            stored_user, stored_hash = line.strip().split(",")
            if stored_user == username and stored_hash == hashed_pw:
                print("Login successful.")
                return username

    print("Invalid credentials.")
    return None


# --- Task Management ---

def add_task(username):
    description = input("Enter task description: ")
    task_file = get_tasks_file(username)

    task_id = 1
    if os.path.exists(task_file):
        with open(task_file, "r") as f:
            lines = f.readlines()
            if lines:
                last_line = lines[-1]
                task_id = int(last_line.split("|")[0]) + 1

    with open(task_file, "a") as f:
        f.write(f"{task_id}|{description}|Pending\n")
    print("Task added successfully.")


def view_tasks(username):
    task_file = get_tasks_file(username)
    if not os.path.exists(task_file):
        print("No tasks found.")
        return

    with open(task_file, "r") as f:
        lines = f.readlines()
        if not lines:
            print("No tasks found.")
            return
        print("\nYour Tasks:")
        for line in lines:
            task_id, desc, status = line.strip().split("|")
            print(f"ID: {task_id} | Description: {desc} | Status: {status}")
    print()


def task_completed(username):
    task_file = get_tasks_file(username)
    task_id = input("Enter task ID to mark as completed: ")

    updated = False
    lines = []
    if os.path.exists(task_file):
        with open(task_file, "r") as f:
            for line in f:
                tid, desc, status = line.strip().split("|")
                if tid == task_id:
                    lines.append(f"{tid}|{desc}|Completed\n")
                    updated = True
                else:
                    lines.append(line)
        with open(task_file, "w") as f:
            f.writelines(lines)

    print("Task marked as completed." if updated else "Task ID not found.")


def delete_task(username):
    task_file = get_tasks_file(username)
    task_id = input("Enter task ID to delete: ")

    deleted = False
    lines = []
    if os.path.exists(task_file):
        with open(task_file, "r") as f:
            for line in f:
                tid, desc, status = line.strip().split("|")
                if tid != task_id:
                    lines.append(line)
                else:
                    deleted = True
        with open(task_file, "w") as f:
            f.writelines(lines)

    print("Task deleted." if deleted else "Task ID not found.")


# --- Main Menu ---

def task_menu(username):
    while True:
        print(f"\n--- Task Menu for {username} ---")
        print("1. Add a Task")
        print("2. View Tasks")
        print("3. Mark Task as Completed")
        print("4. Delete a Task")
        print("5. Logout")

        choice = input("Enter your choice: ")
        if choice == "1":
            add_task(username)
        elif choice == "2":
            view_tasks(username)
        elif choice == "3":
            task_completed(username)
        elif choice == "4":
            delete_task(username)
        elif choice == "5":
            print("Logging out...")
            break
        else:
            print("Invalid choice. Try again.")


# --- Entry Point ---

def main():
    while True:
        print("\n--- Welcome to Task Manager ---")
        print("1. Register")
        print("2. Login")
        print("3. Exit")

        choice = input("Choose an option: ")
        if choice == "1":
            user = register()
            if user:
                task_menu(user)
        elif choice == "2":
            user = login()
            if user:
                task_menu(user)
        elif choice == "3":
            print("Goodbye!")
            break
        else:
            print("Invalid choice. Try again.")


if __name__ == "__main__":
    main()
